# 🔀 Notebook 3: Sharding and Partitioning

When one server can't handle the write load, distribute it across multiple servers. The key is choosing how to split the data.

## Learning Objectives

By the end of this notebook, you'll understand:
- Horizontal vs vertical partitioning
- Choosing effective partition keys
- Avoiding hot spots
- Consistent hashing basics

In [ ]:
import hashlib
import random
from collections import defaultdict
from typing import List, Dict

print("✅ Ready to learn about sharding!")

## 🔀 Horizontal Sharding

In [ ]:
print("🔀 Horizontal Sharding")
print("=" * 60)
print("""
Split ROWS across multiple databases based on a key.

BEFORE (Single DB):
─────────────────────────────────────────────────────────────
┌─────────────────────────────────────────┐
│            All Posts                    │
│  user_id=1, user_id=2, ... user_id=N   │
│         (Bottleneck!)                   │
└─────────────────────────────────────────┘

AFTER (Sharded by user_id):
─────────────────────────────────────────────────────────────
┌─────────────┐  ┌─────────────┐  ┌─────────────┐
│   Shard 0   │  │   Shard 1   │  │   Shard 2   │
│ user_id % 3 │  │ user_id % 3 │  │ user_id % 3 │
│    = 0      │  │    = 1      │  │    = 2      │
└─────────────┘  └─────────────┘  └─────────────┘

• Each shard handles 1/3 of the writes
• Linear scaling: 3 shards = 3x capacity
""")

In [ ]:
class SimpleShardedDB:
    def __init__(self, num_shards: int):
        self.num_shards = num_shards
        self.shards = {i: [] for i in range(num_shards)}
        self.write_counts = {i: 0 for i in range(num_shards)}
    
    def get_shard(self, key: int) -> int:
        return key % self.num_shards
    
    def write(self, user_id: int, data: dict):
        shard_id = self.get_shard(user_id)
        self.shards[shard_id].append({"user_id": user_id, **data})
        self.write_counts[shard_id] += 1
    
    def get_distribution(self) -> dict:
        total = sum(self.write_counts.values())
        return {
            shard_id: {
                "count": count,
                "percentage": (count / total * 100) if total > 0 else 0
            }
            for shard_id, count in self.write_counts.items()
        }

print("🔬 Simulating Writes with Uniform User IDs")
print("=" * 60)

db = SimpleShardedDB(num_shards=4)

for user_id in range(1, 10001):
    db.write(user_id, {"action": "post"})

print("\n📊 Write Distribution (uniform user IDs):")
for shard_id, stats in db.get_distribution().items():
    bar = "█" * int(stats["percentage"] / 2)
    print(f"   Shard {shard_id}: {stats['count']:>5} writes ({stats['percentage']:.1f}%) {bar}")

print("\n✅ Even distribution when keys are uniform!")

## ⚠️ The Hot Spot Problem

In [ ]:
print("⚠️ Bad Partition Key: Country")
print("=" * 60)

class CountryShardedDB:
    def __init__(self):
        self.shards = defaultdict(list)
        self.write_counts = defaultdict(int)
    
    def write(self, country: str, data: dict):
        self.shards[country].append(data)
        self.write_counts[country] += 1

db = CountryShardedDB()

countries = {
    "USA": 330,
    "China": 1400,
    "India": 1380,
    "Brazil": 210,
    "Russia": 140,
    "Japan": 125,
    "Germany": 83,
    "UK": 67,
    "New Zealand": 5,
    "Iceland": 0.3
}

for country, population in countries.items():
    writes = int(population * 10)
    for _ in range(writes):
        db.write(country, {"action": "post"})

total = sum(db.write_counts.values())
print("\n📊 Write Distribution by Country:")
for country in sorted(db.write_counts.keys(), key=lambda x: db.write_counts[x], reverse=True):
    count = db.write_counts[country]
    pct = count / total * 100
    bar = "█" * int(pct / 2)
    print(f"   {country:12}: {count:>6} writes ({pct:>5.1f}%) {bar}")

print("\n❌ China and India get 75% of writes!")
print("   This creates HOT SPOTS - some shards overloaded!")

## 🎯 Choosing Good Partition Keys

In [ ]:
print("🎯 Characteristics of Good Partition Keys")
print("=" * 60)
print("""
GOOD PARTITION KEYS:
─────────────────────────────────────────────────────────────
✅ High cardinality (many unique values)
✅ Uniform distribution of writes
✅ Matches your access patterns
✅ Doesn't change often

Examples:
• user_id - Good for user-centric data
• order_id - Good for e-commerce
• device_id - Good for IoT

─────────────────────────────────────────────────────────────

BAD PARTITION KEYS:
─────────────────────────────────────────────────────────────
❌ Low cardinality (few values)
❌ Skewed distribution
❌ Time-based (all writes go to "current" shard)
❌ Frequently changing

Examples:
• country - Skewed (China >> Iceland)
• status - Low cardinality (pending/active/done)
• timestamp - All "now" writes go to same shard
• celebrity_id - One key gets millions of writes
""")

In [ ]:
print("🎲 Hash-Based Sharding")
print("=" * 60)

def hash_shard(key: str, num_shards: int) -> int:
    hash_value = int(hashlib.md5(str(key).encode()).hexdigest(), 16)
    return hash_value % num_shards

db = SimpleShardedDB(num_shards=4)
db.get_shard = lambda user_id: hash_shard(user_id, 4)

for country, population in countries.items():
    writes = int(population * 10)
    for i in range(writes):
        user_id = f"{country}_{i}"
        shard = hash_shard(user_id, 4)
        db.write_counts[shard] += 1

total = sum(db.write_counts.values())
print("\n📊 Distribution with Hash-Based Sharding:")
for shard_id in sorted(db.write_counts.keys()):
    count = db.write_counts[shard_id]
    pct = count / total * 100
    bar = "█" * int(pct / 2)
    print(f"   Shard {shard_id}: {count:>6} writes ({pct:>5.1f}%) {bar}")

print("\n✅ Hash spreads writes evenly regardless of input distribution!")

## 📐 Vertical Partitioning

In [ ]:
print("📐 Vertical Partitioning")
print("=" * 60)
print("""
Split COLUMNS into different tables/databases based on access patterns.

BEFORE (Monolithic):
─────────────────────────────────────────────────────────────
┌─────────────────────────────────────────────────────────┐
│                        posts                             │
├──────┬─────────┬───────────┬────────────┬───────────────┤
│  id  │ content │ like_cnt  │ view_cnt   │ share_cnt     │
│      │ (write  │ (frequent │ (very      │ (occasional   │
│      │  once)  │  updates) │  frequent) │  updates)     │
└──────┴─────────┴───────────┴────────────┴───────────────┘

Problem: Like/view updates cause locks on content reads!

AFTER (Vertically Partitioned):
─────────────────────────────────────────────────────────────
┌─────────────────┐    ┌──────────────────────────────────┐
│  post_content   │    │         post_metrics             │
├──────┬──────────┤    ├──────┬─────────┬────────┬────────┤
│  id  │ content  │    │  id  │like_cnt │view_cnt│share_ct│
│      │          │    │      │         │        │        │
│ (write once,    │    │ (high-frequency counter updates) │
│  read often)    │    │                                  │
└─────────────────┘    └──────────────────────────────────┘

• Different write patterns = different optimizations
• Content: Optimized for reads (more indexes)
• Metrics: Optimized for writes (minimal indexes)
""")

## 🧪 Quick Quiz

1. **Why is user_id usually a good partition key?**

2. **What's the problem with using timestamp as a partition key?**

3. **When would you use vertical vs horizontal partitioning?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Why user_id is good:")
print("   - High cardinality (millions of users)")
print("   - Users spread writes naturally")
print("   - Matches common access patterns")
print("   - User's data often accessed together")
print()
print("2. Problem with timestamp:")
print("   - All 'current' writes go to same shard")
print("   - Creates sequential hot spot")
print("   - Only latest shard is ever busy")
print()
print("3. Vertical vs Horizontal:")
print("   Vertical: Different access patterns per column")
print("            (content vs counters)")
print("   Horizontal: Same schema, too much data")
print("              (split rows across shards)")

## 📚 Summary

### Key Takeaways

1. **Horizontal sharding** - Split rows across servers
2. **Choose keys wisely** - High cardinality, uniform distribution
3. **Hash for uniformity** - Spreads skewed keys evenly
4. **Vertical partitioning** - Separate by access pattern
5. **Avoid hot spots** - They defeat the purpose of sharding

### Next Up

In **Notebook 4**, we'll learn about queues and load shedding:
- Handling bursty traffic
- Async write patterns
- Graceful degradation